**Objetivo**

Promover os indicadores de alfabetização por unidade federativa da camada Bronze para a Silver.

**Fonte de dados**

- `bronze.uf`

**Destino**

- `silver.uf`

**Granularidade**

- Uma linha por `ano`, `sigla_uf`, `serie` e `rede`.

> As validações de qualidade são informativas, como no notebook de município. Apenas a ausência de colunas obrigatórias impede tecnicamente a execução.

## 0. Configurando sessão Spark

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("bronze_to_silver_uf")
    .config(
        "spark.jars.packages",
        "com.google.cloud.spark:spark-bigquery-with-dependencies_2.13:0.44.2"
    )
    .getOrCreate()
)

spark.conf.set("parentProject", "tech-challenge-fase-2-505123")

:: loading settings :: url = jar:file:/opt/micromamba/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jupyter/.ivy2.5.2/cache
The jars for the packages stored in: /home/jupyter/.ivy2.5.2/jars
com.google.cloud.spark#spark-bigquery-with-dependencies_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-fcdc0a1f-f992-4c2f-bbcf-bfc3c79c3ccd;1.0
	confs: [default]
	found com.google.cloud.spark#spark-bigquery-with-dependencies_2.13;0.44.2 in central
:: resolution report :: resolve 187ms :: artifacts dl 4ms
	:: modules in use:
	com.google.cloud.spark#spark-bigquery-with-dependencies_2.13;0.44.2 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	-----------------------------------------

In [2]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 20)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 100)

## 1. Imports

In [3]:
from pyspark.sql import functions as F

## 2. Geração de parâmetros

In [4]:
par_source_project = "tech-challenge-fase-2-505123"
par_source_bronze_uf = f"{par_source_project}.bronze.uf"
par_source_silver_uf = f"{par_source_project}.silver.uf"

colunas_niveis = [f"proporcao_aluno_nivel_{nivel}" for nivel in range(9)]
colunas_medidas = ["taxa_alfabetizacao", "media_portugues", *colunas_niveis]
colunas_esperadas = [
    "ano", "sigla_uf", "serie", "rede", *colunas_medidas,
    "_ingestao_timestamp", "_fonte"
]

ufs_validas = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO",
    "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI",
    "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO",
]

rede_map = {
    "0": "Total (Federal, Estadual, Municipal e Privada)",
    "1": "Federal",
    "2": "Estadual",
    "3": "Municipal",
    "4": "Privada",
    "5": "Pública (Estadual e Municipal)",
    "6": "Pública (Federal, Estadual e Municipal)",
}

## 3. Leitura dos dados da origem

In [5]:
df_src_uf = (
    spark.read.format("bigquery")
    .option("table", par_source_bronze_uf)
    .load()
)

### 3.1. Validação do contrato de entrada

In [6]:
colunas_ausentes = sorted(set(colunas_esperadas) - set(df_src_uf.columns))
if colunas_ausentes:
    raise ValueError(f"Schema inválido. Colunas ausentes na Bronze: {colunas_ausentes}")

df_uf = df_src_uf.select(*colunas_esperadas)

### 3.2. Diagnóstico dos domínios

In [7]:
for coluna in ["ano", "sigla_uf", "serie", "rede"]:
    print(f"=== Domínio de {coluna} ===")
    (
        df_uf.groupBy(coluna).count()
        .orderBy(F.desc("count"), F.asc_nulls_first(coluna))
        .show(100, truncate=False)
    )

=== Domínio de ano ===


+----+-----+
|ano |count|
+----+-----+
|2024|75   |
|2023|70   |
+----+-----+

=== Domínio de sigla_uf ===
+--------+-----+
|sigla_uf|count|
+--------+-----+
|BA      |7    |
|AL      |6    |
|AM      |6    |
|AP      |6    |
|CE      |6    |
|ES      |6    |
|GO      |6    |
|MA      |6    |
|MG      |6    |
|MS      |6    |
|MT      |6    |
|PA      |6    |
|PB      |6    |
|PE      |6    |
|PI      |6    |
|PR      |6    |
|RN      |6    |
|RO      |6    |
|RS      |6    |
|SC      |6    |
|SE      |6    |
|SP      |6    |
|RJ      |5    |
|TO      |4    |
|AC      |3    |
+--------+-----+

=== Domínio de serie ===
+-----+-----+
|serie|count|
+-----+-----+
|2    |145  |
+-----+-----+

=== Domínio de rede ===
+----+-----+
|rede|count|
+----+-----+
|3   |49   |
|5   |49   |
|2   |46   |
|0   |1    |
+----+-----+



## 4. Transformações

In [8]:
map_rede = F.create_map([F.lit(x) for item in rede_map.items() for x in item])

df_silver_uf = (
    df_uf
    .withColumn("ano", F.col("ano").cast("int"))
    .withColumn("sigla_uf", F.upper(F.trim(F.col("sigla_uf"))))
    .withColumn("serie", F.trim(F.col("serie")).cast("int"))
    .withColumnRenamed("rede", "rede_id")
    .withColumn("rede_id", F.trim(F.col("rede_id").cast("string")))
    .withColumn("rede", map_rede[F.col("rede_id")])
    .withColumn("_ingestao_timestamp", F.col("_ingestao_timestamp").cast("timestamp"))
    .withColumn("_fonte", F.trim(F.col("_fonte")))
)

for coluna in colunas_medidas:
    df_silver_uf = df_silver_uf.withColumn(coluna, F.col(coluna).cast("double"))

### 4.1. Data de carregamento e exclusão de duplicadas

In [9]:
chave = ["ano", "sigla_uf", "serie", "rede_id"]
df_silver_uf_antes_dedup = df_silver_uf
df_silver_uf = (
    df_silver_uf
    .dropDuplicates(chave)
    .withColumn("_silver_timestamp", F.current_timestamp())
)

## 5. Validação da qualidade

In [10]:
print("=== Relatório de Qualidade — silver.uf ===")

qtd_bronze = df_src_uf.count()
qtd_silver = df_silver_uf.count()

# 1) Duplicidade na chave natural
dups_antes = (
    df_silver_uf_antes_dedup.groupBy(*chave).count()
    .filter(F.col("count") > 1).count()
)
dups_depois = (
    df_silver_uf.groupBy(*chave).count()
    .filter(F.col("count") > 1).count()
)
print(f"Chaves duplicadas antes da deduplicação: {dups_antes}")
print(f"Chaves duplicadas após deduplicação: {dups_depois}")

# 2) Domínios das dimensões
uf_invalida = df_silver_uf.filter(
    F.col("sigla_uf").isNotNull() & ~F.col("sigla_uf").isin(*ufs_validas)
).count()
serie_inesperada = df_silver_uf.filter(
    F.col("serie").isNotNull() & (F.col("serie") != 2)
).count()
print(f"Siglas de UF inválidas: {uf_invalida}")
print(f"Séries diferentes de 2: {serie_inesperada}")

print("Códigos de rede não mapeados:")
(
    df_silver_uf.filter(F.col("rede_id").isNotNull() & F.col("rede").isNull())
    .groupBy("rede_id").count().orderBy(F.desc("count"))
    .show(100, truncate=False)
)

# 3) Nulos críticos; níveis são avaliados separadamente porque 2023 não os possui
for coluna in [
    "ano", "sigla_uf", "serie", "rede_id", "taxa_alfabetizacao",
    "media_portugues", "_ingestao_timestamp", "_fonte"
]:
    quantidade = df_silver_uf.filter(F.col(coluna).isNull()).count()
    print(f"Nulos em '{coluna}': {quantidade}")

# 4) Faixas das medidas
taxa_fora_faixa = df_silver_uf.filter(
    (F.col("taxa_alfabetizacao") < 0)
    | (F.col("taxa_alfabetizacao") > 100)
    | F.isnan("taxa_alfabetizacao")
).count()
media_invalida = df_silver_uf.filter(
    (F.col("media_portugues") < 0) | F.isnan("media_portugues")
).count()
print(f"taxa_alfabetizacao fora de [0, 100] ou NaN: {taxa_fora_faixa}")
print(f"media_portugues negativa ou NaN: {media_invalida}")

for coluna in colunas_niveis:
    quantidade = df_silver_uf.filter(
        F.col(coluna).isNotNull()
        & ((F.col(coluna) < 0) | (F.col(coluna) > 100) | F.isnan(coluna))
    ).count()
    print(f"{coluna} preenchida e fora de [0, 100] ou NaN: {quantidade}")

# 5) Completude conjunta e soma das proporções
qtd_niveis_preenchidos = sum(
    F.when(F.col(coluna).isNotNull(), F.lit(1)).otherwise(F.lit(0))
    for coluna in colunas_niveis
)
soma_niveis = sum(F.coalesce(F.col(coluna), F.lit(0.0)) for coluna in colunas_niveis)

niveis_parcialmente_preenchidos = df_silver_uf.filter(
    (qtd_niveis_preenchidos > 0) & (qtd_niveis_preenchidos < 9)
).count()
soma_niveis_invalida = df_silver_uf.filter(
    (qtd_niveis_preenchidos == 9)
    & ((soma_niveis < 99.9) | (soma_niveis > 100.1))
).count()
print(f"Linhas com apenas parte dos nove níveis preenchida: {niveis_parcialmente_preenchidos}")
print(f"Linhas completas cuja soma dos níveis está fora de [99.9, 100.1]: {soma_niveis_invalida}")

print("Preenchimento dos níveis por ano:")
(
    df_silver_uf.withColumn("qtd_niveis_preenchidos", qtd_niveis_preenchidos)
    .groupBy("ano", "qtd_niveis_preenchidos").count()
    .orderBy("ano", "qtd_niveis_preenchidos")
    .show(100, truncate=False)
)

# 6) Cobertura de UFs por ano (ausência pode refletir não participação)
print("Quantidade de UFs distintas por ano:")
(
    df_silver_uf.groupBy("ano")
    .agg(F.countDistinct("sigla_uf").alias("qtd_ufs"))
    .orderBy("ano").show()
)

print(f"Linhas Bronze: {qtd_bronze} -> Linhas Silver: {qtd_silver}")

=== Relatório de Qualidade — silver.uf ===


Chaves duplicadas antes da deduplicação: 0
Chaves duplicadas após deduplicação: 0
Siglas de UF inválidas: 0
Séries diferentes de 2: 0
Códigos de rede não mapeados:
+-------+-----+
|rede_id|count|
+-------+-----+
+-------+-----+

Nulos em 'ano': 0
Nulos em 'sigla_uf': 0
Nulos em 'serie': 0
Nulos em 'rede_id': 0
Nulos em 'taxa_alfabetizacao': 0
Nulos em 'media_portugues': 0
Nulos em '_ingestao_timestamp': 0
Nulos em '_fonte': 0
taxa_alfabetizacao fora de [0, 100] ou NaN: 0
media_portugues negativa ou NaN: 0
proporcao_aluno_nivel_0 preenchida e fora de [0, 100] ou NaN: 0
proporcao_aluno_nivel_1 preenchida e fora de [0, 100] ou NaN: 0
proporcao_aluno_nivel_2 preenchida e fora de [0, 100] ou NaN: 0
proporcao_aluno_nivel_3 preenchida e fora de [0, 100] ou NaN: 0
proporcao_aluno_nivel_4 preenchida e fora de [0, 100] ou NaN: 0
proporcao_aluno_nivel_5 preenchida e fora de [0, 100] ou NaN: 0
proporcao_aluno_nivel_6 preenchida e fora de [0, 100] ou NaN: 0
proporcao_aluno_nivel_7 preenchida e fora

## 6. Armazenamento no BigQuery

In [11]:
(
    df_silver_uf.write.format("bigquery")
    .option("table", par_source_silver_uf)
    .option("writeMethod", "direct")
    .option("clusteredFields", "ano,sigla_uf,rede_id")
    .mode("overwrite")
    .save()
)

26/08/24 01:44:50 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                